In [15]:
import torch 
import torch.nn as nn
import torchvision
import torch.optim as optim
import torchvision.models as models
import torchvision.transforms as transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [54]:
# Data preparation 
# Transform: resize to 224x224, grayscale->RGB
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # Resize first
    transforms.Grayscale(num_output_channels=3),  #
    transforms.ToTensor()
])

train_dataset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset  = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

# Create synthetic "not a number" dataset from random noise
class NoiseDataset(torch.utils.data.Dataset):
    def __init__(self, length=6000, transform=None):
        self.length = length
        self.transform = transform
    def __len__(self):
        return self.length
    def __getitem__(self, idx):
        img = torch.rand(1, 28, 28)  # random noise
        if self.transform:
            img = self.transform(img)
        label = 10  # class index for "not a number"
        return img, label

noise_train = NoiseDataset(length=len(train_dataset)//10, transform=transform)
noise_test  = NoiseDataset(length=len(test_dataset)//10, transform=transform)

# Combine datasets
train_dataset = torch.utils.data.ConcatDataset([train_dataset, noise_train])
test_dataset  = torch.utils.data.ConcatDataset([test_dataset, noise_test])

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader  = torch.utils.data.DataLoader(test_dataset, batch_size=64, shuffle=False)

print(f"Training samples: {train_loader}, Testing samples: {train_loader}")

(image, label) = iter(train_loader)


Training samples: <torch.utils.data.dataloader.DataLoader object at 0x7d4f11a8f8c0>, Testing samples: <torch.utils.data.dataloader.DataLoader object at 0x7d4f11a8f8c0>


TypeError: pic should be PIL Image or ndarray. Got <class 'torch.Tensor'>

In [13]:
model = models.mobilenet_v3_small(pretrained=False)
model.classifier[3] = nn.Linear(model.classifier[3].in_features, 11)
model = model.to(device)


/home/jha/Projects/Computer Vision/.venv/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/jha/Projects/Computer Vision/.venv/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


In [21]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


In [32]:
for epoch in range(5):  # try more (e.g. 20) for better accuracy
    model.train()
    running_loss = 0.0
    print("Starting the epoch")
    for idx in range(len(train_loader)):
        print(train_loader[idx])
    # for (images, labels) in train_loader:
    #     print(f"Starting output for images of shape")
    #     images, labels = images.to(device), labels.to(device)
    #     print(labels)
    #     optimizer.zero_grad()
    #     outputs = model(images)
    #     loss = criterion(outputs, labels)
    #     loss.backward()
    #     optimizer.step()
        
    #     running_loss += loss.item()
    
    print(f"Epoch {epoch+1}, Loss: {running_loss/len(train_loader):.4f}")

Starting the epoch


TypeError: 'DataLoader' object is not subscriptable

In [ ]:
model.eval()
correct, total = 0, 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Test Accuracy: {100 * correct / total:.2f}%")
